## Importing data and libraries

In [ ]:
import pandas as pd 
import numpy as np
import optuna
import xgboost as xgb 
import lightgbm as lgb
import catboost as cb
from optbinning import OptimalBinning
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import plotly.graph_objects as go
from sklearn.model_selection import GroupKFold

In [ ]:
print("Reading data ...")
train_org = pd.read_csv("playground-series-s6e5\\train.csv")
test_org = pd.read_csv("playground-series-s6e5\\test.csv")
submissions_org = pd.read_csv("playground-series-s6e5\\sample_submission.csv")

train_org.shape, test_org.shape, submissions_org.shape

## Configurations
- Binning
- Feature Engineering

In [ ]:
AUC_BENCHMARK = 0.9505
F_ENGG = True
MAX_BIN = 20
MIN_BIN = 0.001
SHOW_WOE = False

## Research & Feature Engineering

In [ ]:
def engineer_race_features(df, activate=True):

    if not activate:
        return df

    df = df.copy()

    eps = 1e-5

    # =========================================================
    # BASIC PROGRESS FEATURES
    # =========================================================

    df['Progress_Per_Lap_engg'] = (
        df['RaceProgress'] / (df['LapNumber'] + eps)
    )

    df['Remaining_RaceProgress_engg'] = (
        1 - df['RaceProgress']
    )

    df['Remaining_Laps_Ratio_engg'] = (
        (1 - df['RaceProgress']) /
        (df['LapNumber'] + eps)
    )

    # =========================================================
    # DEGRADATION FEATURES
    # =========================================================

    df['Deg_Per_Lap_engg'] = (
        df['Cumulative_Degradation'] /
        (df['LapNumber'] + eps)
    )

    df['Deg_Per_TyreLife_engg'] = (
        df['Cumulative_Degradation'] /
        (df['TyreLife'] + eps)
    )

    df['Deg_x_TyreLife_engg'] = (
        df['Cumulative_Degradation'] *
        df['TyreLife']
    )

    df['Deg_x_Progress_engg'] = (
        df['Cumulative_Degradation'] *
        df['RaceProgress']
    )

    df['Deg_Acceleration_engg'] = (
        df['Cumulative_Degradation'] /
        (df['RaceProgress'] + eps)
    )

    # =========================================================
    # PACE FEATURES
    # =========================================================

    df['Pace_Tyre_Sensitivity_engg'] = (
        df['LapTime_Delta'] /
        (df['TyreLife'] + eps)
    )

    df['Pace_Per_Position_engg'] = (
        df['LapTime_Delta'] /
        (df['Position'] + eps)
    )

    df['LapTime_x_TyreLife_engg'] = (
        df['LapTime_Delta'] *
        df['TyreLife']
    )

    df['LapTime_x_Deg_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['LapTime_x_Progress_engg'] = (
        df['LapTime_Delta'] *
        df['RaceProgress']
    )

    df['Pace_Drop_Flag_engg'] = (
        df['LapTime_Delta'] > 0
    ).astype(int)

    df['Extreme_Pace_Drop_engg'] = (
        df['LapTime_Delta'] > df['LapTime_Delta'].quantile(0.90)
    ).astype(int)

    # =========================================================
    # TYRE FEATURES
    # =========================================================

    df['TyreLife_Per_Lap_engg'] = (
        df['TyreLife'] /
        (df['LapNumber'] + eps)
    )

    df['TyreLife_Per_Progress_engg'] = (
        df['TyreLife'] /
        (df['RaceProgress'] + eps)
    )

    df['TyreLife_x_Progress_engg'] = (
        df['TyreLife'] *
        df['RaceProgress']
    )

    df['Fresh_Tyre_Flag_engg'] = (
        df['TyreLife'] <= 5
    ).astype(int)

    df['Medium_Tyre_Flag_engg'] = (
        (df['TyreLife'] > 5) &
        (df['TyreLife'] <= 20)
    ).astype(int)

    df['Old_Tyre_Flag_engg'] = (
        df['TyreLife'] > 20
    ).astype(int)

    # =========================================================
    # POSITION FEATURES
    # =========================================================

    df['Losing_Ground_engg'] = (
        df['Position_Change'] < 0
    ).astype(int)

    df['Gaining_Ground_engg'] = (
        df['Position_Change'] > 0
    ).astype(int)

    df['Position_x_Progress_engg'] = (
        df['Position'] *
        df['RaceProgress']
    )

    df['Position_x_TyreLife_engg'] = (
        df['Position'] *
        df['TyreLife']
    )

    df['Position_Change_Intensity_engg'] = (
        df['Position_Change'] /
        (df['LapNumber'] + eps)
    )

    df['Bad_Position_Flag_engg'] = (
        df['Position'] > 10
    ).astype(int)

    df['Podium_Position_Flag_engg'] = (
        df['Position'] <= 3
    ).astype(int)

    # =========================================================
    # PIT WINDOW FEATURES
    # =========================================================

    df['Potential_Pit_Window_engg'] = (
        (df['TyreLife'] > 15) &
        (df['RaceProgress'] > 0.25)
    ).astype(int)

    df['Late_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] > 0.70)
    ).astype(int)

    df['Early_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] < 0.25)
    ).astype(int)

    # =========================================================
    # INTERACTION FEATURES
    # =========================================================

    df['Wear_Pace_Impact_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['Wear_Position_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position']
    )

    df['Wear_Position_Change_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position_Change']
    )

    df['TyreLife_Position_Interaction_engg'] = (
        df['TyreLife'] *
        df['Position']
    )

    df['TyreLife_Lap_Interaction_engg'] = (
        df['TyreLife'] *
        df['LapNumber']
    )

    df['TyreLife_Stint_Interaction_engg'] = (
        df['TyreLife'] *
        df['Stint']
    )

    df['Lap_Position_Interaction_engg'] = (
        df['LapNumber'] *
        df['Position']
    )

    # =========================================================
    # STINT FEATURES
    # =========================================================

    df['Is_First_Stint_engg'] = (
        df['Stint'] == 1
    ).astype(int)

    df['Is_Second_Stint_engg'] = (
        df['Stint'] == 2
    ).astype(int)

    df['Is_ThirdPlus_Stint_engg'] = (
        df['Stint'] >= 3
    ).astype(int)

    df['Stint_x_Progress_engg'] = (
        df['Stint'] *
        df['RaceProgress']
    )

    df['Stint_x_TyreLife_engg'] = (
        df['Stint'] *
        df['TyreLife']
    )

    # =========================================================
    # CATEGORICAL COMBINATIONS
    # =========================================================

    df['Year_Stint_engg'] = (
        df['Year'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Year_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Race_Stint_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Driver_Compound_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Driver_Race_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Race'].astype(str)
    )

    df['Compound_Stint_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Position_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Position'].astype(str)
    )

    df['Race_Compound_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Race_Year_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Driver_Stint_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    return df


train_df = engineer_race_features(train_org, activate=F_ENGG)
test_df = engineer_race_features(test_org, activate=F_ENGG)

t_col = "PitNextLap"
ID = "id"
c_cols = list(test_df.select_dtypes(include='object').columns)
n_cols = list(test_df.select_dtypes(exclude='object').columns)

print("--" * 20, "Categorical Columns", "--" * 20)
print(train_df[c_cols].head())
print("--" * 20, "Numerical Columns", "--" * 20)
print(train_df[n_cols].head())

## Optbinning / WOE / IV
- Categorical binning
- Numerical binning
- Top features by IV
- WOE Trend for each feature

In [ ]:
class OptimalBinner:

    def __init__(
        self, cat_cols: list, num_cols: list, max_n_bins: int = 5, min_bin_size: float = 0.05):
        self.cat_cols = cat_cols
        self.num_cols = num_cols
        self.max_n_bins = max_n_bins
        self.min_bin_size = min_bin_size
        self.binners = {}


    def fit(self, X, y, verbos=True):
        for col in X.columns:
            if verbos:
                print(f"Fitting: {col}")
            if col in self.cat_cols:
                optb = OptimalBinning(
                    name=col,
                    dtype="categorical",
                    max_n_bins=self.max_n_bins,
                    min_bin_size=self.min_bin_size
                )

            else:
                optb = OptimalBinning(
                    name=col,
                    dtype="numerical",
                    max_n_bins=self.max_n_bins,
                    min_bin_size=self.min_bin_size
                )
            # Fit binning
            optb.fit(X[col], y)
            self.binners[col] = optb
        return self


    def transform(self, X, metric="woe"):
        X_transformed = pd.DataFrame(index=X.index)
        for col in X.columns:
            optb = self.binners[col]
            X_transformed[col] = optb.transform(
                X[col],
                metric=metric,
                metric_missing="empirical",
                metric_special="empirical"
            )
        return X_transformed


    def get_iv_summary(self):
        iv_data = []
        for col, optb in self.binners.items():
            iv = optb.binning_table.build()["IV"].sum()
            iv_data.append({
                "feature": col,
                "iv": iv
            })
        return (
            pd.DataFrame(iv_data)
            .sort_values("iv", ascending=False)
            .reset_index(drop=True)
        )

    def get_binning_table(self, col):
        return self.binners[col].binning_table.build()



class LGB_XGB_CAT_Model:

    def __init__(self, lgb_params=None, xgb_params=None, cb_params=None):

        self.lgb_model = None
        self.xgb_model = None
        self.cb_model = None

        self.lgb_params = lgb_params
        self.xgb_params = xgb_params
        self.cb_params = cb_params

    def train(self, X_train, y_train):

        if type(self.lgb_params) != type(None):
            print("Training LGB model...")
            self.lgb_model = lgb.LGBMClassifier(
                **self.lgb_params
            ).fit(
                X_train,
                y_train
            )

        if type(self.xgb_params) != type(None):
            print("Training XGB model...")
            self.xgb_model = xgb.XGBClassifier(
                **self.xgb_params
            ).fit(
                X_train,
                y_train
            )

        if type(self.cb_params) != type(None):
            print("Training CatBoost model...")
            self.cb_model = cb.CatBoostClassifier(
                **self.cb_params
            ).fit(
                X_train,
                y_train
            )
        print("Models trained successfully!")

    def predict_proba(self, X):
        probs = {}
        if type(self.lgb_params) != type(None):
            probs['lgb'] = (
                self.lgb_model
                .predict_proba(X)[:, 1]
            )

        if type(self.xgb_params) != type(None):
            probs['xgb'] = (
                self.xgb_model
                .predict_proba(X)[:, 1]
            )

        if type(self.cb_params) != type(None):
            probs['cb'] = (
                self.cb_model
                .predict_proba(X)[:, 1]
            )
        return probs

    def roc_auc_score(self, X_test, y_test):
        roc_auc_dict = {}
        if type(self.lgb_params) != type(None):
            roc_auc_dict['lgb'] = roc_auc_score(y_test, self.lgb_model.predict_proba(X_test)[:, 1])

        if type(self.xgb_params) != type(None):
            roc_auc_dict['xgb'] = roc_auc_score(y_test, self.xgb_model.predict_proba(X_test)[:, 1])

        if type(self.cb_params) != type(None):
            roc_auc_dict['cb'] = roc_auc_score(y_test, self.cb_model.predict_proba(X_test)[:, 1])

        return roc_auc_dict

    def cross_validation(self, X, y, groups, n_splits=5):
        gkf = GroupKFold(
            n_splits=n_splits
        )
        cv_scores = {
            "lgb": [],
            "xgb": [],
            "cb": []
        }

        for train_idx, valid_idx in gkf.split(X, y, groups=groups):
            X_train_cv = X.iloc[train_idx]
            X_valid_cv = X.iloc[valid_idx]

            y_train_cv = y.iloc[train_idx]
            y_valid_cv = y.iloc[valid_idx]

            if type(self.lgb_params) != type(None):

                model_lgb = lgb.LGBMClassifier(**self.lgb_params)
                model_lgb.fit(X_train_cv, y_train_cv)
                y_prob_lgb = model_lgb.predict_proba(X_valid_cv)[:, 1]
                auc_lgb = roc_auc_score(y_valid_cv, y_prob_lgb)
                cv_scores["lgb"].append(auc_lgb)

            if type(self.xgb_params) != type(None):
                model_xgb = xgb.XGBClassifier(**self.xgb_params)
                model_xgb.fit(X_train_cv, y_train_cv)
                y_prob_xgb = model_xgb.predict_proba(X_valid_cv)[:, 1]
                auc_xgb = roc_auc_score(y_valid_cv, y_prob_xgb)
                cv_scores["xgb"].append(auc_xgb)

            if type(self.cb_params) != type(None):
                model_cb = cb.CatBoostClassifier(**self.cb_params)
                model_cb.fit(X_train_cv, y_train_cv, verbose=False)
                y_prob_cb = model_cb.predict_proba(X_valid_cv)[:, 1]
                auc_cb = roc_auc_score(y_valid_cv, y_prob_cb)
                cv_scores["cb"].append(auc_cb)

        final_scores = {}

        if len(cv_scores["lgb"]) > 0:
            final_scores["lgb"] = np.mean(cv_scores["lgb"])

        if len(cv_scores["xgb"]) > 0:
            final_scores["xgb"] = np.mean(cv_scores["xgb"])

        if len(cv_scores["cb"]) > 0:
            final_scores["cb"] = np.mean(cv_scores["cb"])

        return final_scores



X = train_df.drop(columns=[t_col], axis='columns')
y = train_df[t_col]

X_test = test_df.copy()

groups = (
    train_df["Race"].astype(str)
    + "_"
    + train_df["Year"].astype(str)
)

if MAX_BIN == None and MIN_BIN == None:
    max_n_bins = [5, 10, 15, 20, 25]
    min_bin_size = [0.001, 0.05, 0.06, 0.07]
    best_bin_range = {}
    best_roc_auc = 0.0

    print(
        "Searching for best max_bins and min_bins combination..."
    )

    for max_bin in max_n_bins:
        for min_bin in min_bin_size:
            print(
                f"Fitting with max_bin = {max_bin} and min_bin = {min_bin}..."
            )
            bin_obj = OptimalBinner(
                cat_cols=c_cols,
                num_cols=n_cols,
                max_n_bins=max_bin,
                min_bin_size=min_bin
            )
            bin_obj.fit(
                X,
                y,
                verbos=False
            )
            X_train_woe = bin_obj.transform(
                X,
                metric="woe"
            )
            model_base = LGB_XGB_CAT_Model(
                xgb_params={}
            )
            auc_dict = model_base.cross_validation(
                X=X_train_woe,
                y=y,
                groups=groups,
                n_splits=5
            )
            auc = auc_dict['xgb']
            if best_roc_auc < auc:
                best_roc_auc = auc
                best_bin_range['max_bin'] = max_bin
                best_bin_range['min_bin'] = min_bin

            print(
                "best_bin_range",
                best_bin_range,
                "auc",
                auc,
                "best_roc_auc",
                best_roc_auc
            )

    MAX_BIN = best_bin_range['max_bin']
    MIN_BIN = best_bin_range['min_bin']

bin_obj = OptimalBinner(
    cat_cols=c_cols,
    num_cols=n_cols,
    max_n_bins=MAX_BIN,
    min_bin_size=MIN_BIN
)

bin_obj.fit(
    X,
    y,
    verbos=False
)

X_train_woe = bin_obj.transform(
    X,
    metric="woe"
)

X_test_woe = bin_obj.transform(
    X_test,
    metric="woe"
)

print(X_train_woe.shape)
print(X_test_woe.shape)

In [ ]:
iv_df = bin_obj.get_iv_summary()

top_iv = (
    iv_df
    .sort_values("iv", ascending=True)
    .reset_index(drop=True)
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_iv["iv"],
        y=top_iv["feature"],
        orientation="h",
        text=top_iv["iv"].round(3),
        textposition="outside"
    )
)

fig.update_layout(
    title="Interactive Information Value (IV) Analysis",
    xaxis_title="Information Value",
    yaxis_title="Feature",
    template="plotly_white",
    height=max(500, len(top_iv) * 35),
    hovermode="closest",
    showlegend=False
)

fig.show()

## WOE Trends

In [ ]:
def plot_woe_trend(bin_obj, feature_name, figsize_height=500):

    bt = (
        bin_obj
        .binners[feature_name]
        .binning_table
        .build()
    )

    bt = bt[
        ~bt["Bin"].astype(str).isin(
            ["Totals", "Special", "Missing"]
        )
    ].copy()


    fig = go.Figure()

    fig.add_trace(

        go.Scatter(
            x=bt["Bin"].astype(str),
            y=bt["WoE"],

            mode="lines+markers",

            name="WOE",

            yaxis="y1",

            hovertemplate=
            "<b>Bin:</b> %{x}<br>" +
            "<b>WOE:</b> %{y:.4f}<extra></extra>"
        )
    )


    fig.add_trace(

        go.Bar(
            x=bt["Bin"].astype(str),
            y=bt["Event rate"],

            name="Event Rate",

            yaxis="y2",

            opacity=0.5,

            hovertemplate=
            "<b>Bin:</b> %{x}<br>" +
            "<b>Event Rate:</b> %{y:.4f}<extra></extra>"
        )
    )

    fig.update_layout(

        title=f"WOE Trend Analysis: {feature_name}",

        xaxis=dict(
            title="Bins"
        ),

        yaxis=dict(
            title="WOE",
            side="left"
        ),

        yaxis2=dict(
            title="Event Rate",
            overlaying="y",
            side="right"
        ),

        template="plotly_white",

        hovermode="x unified",

        height=figsize_height,

        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.show()

if SHOW_WOE:
    for feature in X_train_woe.columns:
        if feature.lower() == 'driver':
            continue
        plot_woe_trend(
            bin_obj,
            feature_name=feature
        )

## Train models - Default params
- LightGBM 
- XGBoost

In [ ]:
SPW = len(y[y == 0]) / len(y[y == 1])

X_train, X_test, y_train, y_test = train_test_split(X_train_woe, y, test_size=0.2, random_state=42)
model_base = LGB_XGB_Model(xgb_params={"scale_pos_weight": SPW})
model_base.train(X_train, y_train)

print(model_base.roc_auc_score(X_test, y_test))
print(f"Base model performance is {'***** better *****' if max(model_base.roc_auc_score(X_test, y_test).values()) > AUC_BENCHMARK else '!!!!!!! worse !!!!!!!'} than benchmark {AUC_BENCHMARK}")

In [ ]:
best_params_cb = {'depth': 6, 'min_data_in_leaf': 1, 'random_strength': 1, 'subsample': 0.800000011920929, 'colsample_bylevel': 0.9890729302630086, 'l2_leaf_reg': 3, 'learning_rate': 0.1258769929409027, 'iterations': 1000}
model_base = LGB_XGB_Model(xgb_params={"scale_pos_weight": SPW})
model_base.train(X_train_woe, y)

In [ ]:
predictions = model_base.predict_proba(X_test_woe)
ids = test_df[ID].values
submission = pd.DataFrame({f"{ID}": ids, f"{t_col}": predictions['xgb']})
submission.to_csv("XGB_New_Features_Binning_Optimal_23_05_2026.csv", index=False)

In [ ]:
submission